# Script Transliteration for Turkic Languages

This notebook demonstrates bidirectional script transliteration for Turkic languages using the TurkicNLP toolkit.

**Chapter reference:** Chapter 3 (Scripts, Encoding, and Orthographic Engineering)

**Coverage:**
- Script detection (Latin, Cyrillic, Perso-Arabic, Old Turkic Runic)
- Bidirectional transliteration (Cyrillic ↔ Latin, Arabic ↔ Latin)
- Unicode normalization and homoglyph handling
- Turkish İ/I case handling
- Text normalization pipeline: diacritic restoration, apostrophe handling
- Cross-script text processing

**Languages covered:** Kazakh, Uzbek, Azerbaijani, Tatar, Uyghur, Turkmen, Karakalpak, Crimean Tatar, Ottoman Turkish

In [ ]:
# Install TurkicNLP with full support
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import unicodedata
import turkicnlp
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

## 1. Script Detection

Automatically identify which script a text uses (Latin, Cyrillic, Perso-Arabic, or Old Turkic Runic).

In [ ]:
# Examples in different scripts
examples = {
    "Kazakh (Cyrillic)": "Мен Алматыда тұрамын.",
    "Kazakh (Latin)": "Men Almatyda turamyn.",
    "Uzbek (Cyrillic)": "Ўзбекистон бир бўлак давлат.",
    "Uzbek (Latin)": "Oʻzbekiston bir boʻlak davlat.",
    "Uyghur (Arabic)": "مەن ئالماتيدا تۇرامىن.",
    "Turkish (Latin)": "Türkiye'de Türkçe konuşuyorum.",
    "Ottoman Turkish (Arabic)": "تركيه‌ده تركجه كونوشيورم.",
}

print("Script Detection:")
print("=" * 50)
for lang, text in examples.items():
    script = detect_script(text)
    print(f"{lang:<25} → {script.name}")
    print(f"  Text: {text}")
    print()

## 2. Cyrillic ↔ Latin Transliteration

Bidirectional transliteration for Central Asian languages that transitioned from Cyrillic to Latin.

In [ ]:
# Kazakh: Cyrillic → Latin
print("Kazakh: Cyrillic ↔ Latin (2021 official alphabet)")
print("=" * 50)

kaz_cyrl = "Мен Алматыда университетте оқимын."
print(f"Cyrillic: {kaz_cyrl}")

# Create transliterators for each direction
t_cyrl_to_latn = Transliterator("kaz", source=Script.CYRILLIC, target=Script.LATIN)
kaz_latn = t_cyrl_to_latn.transliterate(kaz_cyrl)
print(f"Latin:    {kaz_latn}")

# Reverse: Latin → Cyrillic
t_latn_to_cyrl = Transliterator("kaz", source=Script.LATIN, target=Script.CYRILLIC)
kaz_restored = t_latn_to_cyrl.transliterate(kaz_latn)
print(f"Restored: {kaz_restored}")
print(f"Match: {kaz_cyrl == kaz_restored}")
print()

In [ ]:
# Uzbek: Cyrillic → Latin (1995 official alphabet)
print("Uzbek: Cyrillic ↔ Latin (1995 official alphabet)")
print("=" * 50)

uzb_cyrl = "Ўзбекистон республикаси ўз достлигини қўллайди."
print(f"Cyrillic: {uzb_cyrl}")

t_uz_cyrl = Transliterator("uzb", source=Script.CYRILLIC, target=Script.LATIN)
uzb_latn = t_uz_cyrl.transliterate(uzb_cyrl)
print(f"Latin:    {uzb_latn}")

t_uz_latn = Transliterator("uzb", source=Script.LATIN, target=Script.CYRILLIC)
uzb_restored = t_uz_latn.transliterate(uzb_latn)
print(f"Restored: {uzb_restored}")
print()

In [ ]:
# Azerbaijani: Latin and Cyrillic coexistence
print("Azerbaijani: Cyrillic ↔ Latin (1991 transition, still mixed in practice)")
print("=" * 50)

aze_latn = "Azərbaycanda gül şəxsiyyətinin simvoludur."
aze_cyrl = "Азәрбајҹанда гүл шәхсијәтінің символудур."

print(f"Latin:    {aze_latn}")
print(f"Cyrillic: {aze_cyrl}")

# Convert Latin to Cyrillic
t_az = Transliterator("aze", source=Script.LATIN, target=Script.CYRILLIC)
aze_from_latn = t_az.transliterate(aze_latn)
print(f"\nLatin → Cyrillic: {aze_from_latn}")
print()

## 3. Perso-Arabic ↔ Latin Transliteration

Transliteration for Uyghur and Ottoman Turkish using Perso-Arabic script.

In [ ]:
# Uyghur: Arabic → Latin (ULY, Uyghur Latin Yéziqi)
print("Uyghur: Perso-Arabic ↔ Latin (ULY - Uyghur Latin Yéziqi)")
print("=" * 50)

uig_arab = "مەن ئالماتيدا تۇرامىن."
print(f"Arabic: {uig_arab}")

t_uig = Transliterator("uig", source=Script.ARABIC, target=Script.LATIN)
uig_latn = t_uig.transliterate(uig_arab)
print(f"Latin:  {uig_latn}")

# Note: ZWNJ (Zero-Width Non-Joiner) preservation
print(f"\nNote: ZWNJ characters (U+200C) mark morpheme boundaries and are preserved.")
print()

In [ ]:
# Ottoman Turkish: Arabic → Latin
print("Ottoman Turkish: Arabic → Latin (academic convention)")
print("=" * 50)

ota_arab = "ترکيه‌ده ترکجه کونوشيورم."
print(f"Arabic: {ota_arab}")

t_ota = Transliterator("ota", source=Script.ARABIC, target=Script.LATIN)
ota_latn = t_ota.transliterate(ota_arab)
print(f"Latin:  {ota_latn}")
print("\nNote: Ottoman Turkish transliteration is one-way (Arabic → Latin only).")
print()

## 4. Unicode Normalization and Homoglyph Handling

Handle Unicode edge cases: the Turkish İ/I problem, Cyrillic/Latin homoglyphs, and NFC normalization.

In [ ]:
# Unicode NFC normalization (always the first step)
print("Unicode NFC Normalization")
print("=" * 50)

# Example: Turkish ş as precomposed vs. decomposed
composed = "Spor"  # NFC (precomposed)
decomposed = "S\u0323por"  # NFD (s + combining cedilla)

print(f"Precomposed (NFC):  '{composed}' (length: {len(composed)})")
print(f"Decomposed (NFD):   '{decomposed}' (length: {len(decomposed)})")
print(f"Are they equal? {composed == decomposed}")

# Normalize both to NFC
norm_composed = unicodedata.normalize('NFC', composed)
norm_decomposed = unicodedata.normalize('NFC', decomposed)
print(f"\nAfter NFC normalization:")
print(f"Both equal? {norm_composed == norm_decomposed}")
print()

In [ ]:
# The Turkish İ/I problem
print("Turkish İ/I Case Handling")
print("=" * 50)

print("Turkish has 4 distinct characters where English has 2:")
print(f"  İ (U+0130) - Capital dotted I")
print(f"  I (U+0049) - Capital dotless I")
print(f"  i (U+0069) - Lowercase dotted i")
print(f"  ı (U+0131) - Lowercase dotless ı")
print()

# Demonstrate the problem with standard Python methods
test_words = ["İstanbul", "Isparta"]

print("Problem: Python's str.lower() is locale-dependent")
for word in test_words:
    # This may be wrong on non-Turkish systems
    lowered = word.lower()
    print(f"  '{word}'.lower() → '{lowered}'")
print()

# Correct approach: use Turkish-aware functions
def turkish_lower(text: str) -> str:
    """Turkish-aware case conversion to lowercase."""
    return text.replace("İ", "i").replace("I", "ı").lower()

def turkish_upper(text: str) -> str:
    """Turkish-aware case conversion to uppercase."""
    return text.replace("i", "İ").replace("ı", "I").upper()

print("Correct approach: Turkish-aware functions")
for word in test_words:
    lowered = turkish_lower(word)
    uppered = turkish_upper(lowered)
    print(f"  '{word}' → lower: '{lowered}' → upper: '{uppered}'")
print()

In [ ]:
# Detect Cyrillic/Latin homoglyphs
print("Cyrillic/Latin Homoglyph Detection")
print("=" * 50)

homoglyphs = {
    'a': ('Latin', 'U+0061'),
    'а': ('Cyrillic', 'U+0430'),
    'e': ('Latin', 'U+0065'),
    'е': ('Cyrillic', 'U+0435'),
    'o': ('Latin', 'U+006F'),
    'о': ('Cyrillic', 'U+043E'),
    'p': ('Latin', 'U+0070'),
    'р': ('Cyrillic', 'U+0440'),
}

print("\nHomoglyph pairs (visually identical, different codepoints):")
for char, (script, codepoint) in sorted(homoglyphs.items()):
    print(f"  '{char}' {codepoint} ({script})")

def detect_homoglyphs(text: str):
    """Find positions of Cyrillic/Latin homoglyphs in text."""
    homoglyph_chars = {'а', 'е', 'о', 'р'}  # Cyrillic versions
    results = []
    for i, char in enumerate(text):
        if char in homoglyph_chars:
            results.append((i, char, ord(char)))
    return results

# Example: mixed-script text
mixed_text = "Almatа tурамы"  # Contains Cyrillic 'а' and 'у'
print(f"\nDetecting homoglyphs in mixed text: '{mixed_text}'")
homoglyphs_found = detect_homoglyphs(mixed_text)
for pos, char, codepoint in homoglyphs_found:
    print(f"  Position {pos}: '{char}' (U+{codepoint:04X})")
print()

## 5. Text Normalization Pipeline

Complete preprocessing pipeline: Unicode normalization, script detection, transliteration, and diacritic restoration.

In [ ]:
def normalize_turkic_text(text: str, target_script: Script = Script.LATIN, lang: str = "kaz") -> str:
    """
    Full preprocessing pipeline for Turkic text.
    
    Steps:
    1. Unicode NFC normalization
    2. Script detection
    3. Transliteration to target script (if needed)
    4. Homoglyph detection and warning
    """
    
    # Step 1: Unicode normalization
    normalized = unicodedata.normalize('NFC', text)
    if normalized != text:
        print(f"  [Unicode] Normalized (NFC)")
    
    # Step 2: Script detection
    detected_script = detect_script(text)
    print(f"  [Script] Detected: {detected_script.name}")
    
    # Step 3: Transliteration if script doesn't match target
    if detected_script != target_script:
        try:
            t = Transliterator(lang, source=detected_script, target=target_script)
            normalized = t.transliterate(normalized)
            print(f"  [Translit] {detected_script.name} → {target_script.name}")
        except Exception as e:
            print(f"  [Translit] Skipped: {e}")
    
    # Step 4: Homoglyph detection
    homos = detect_homoglyphs(normalized)
    if homos:
        print(f"  [Warning] Found {len(homos)} potential Cyrillic/Latin homoglyphs")
    
    return normalized

# Example 1: Kazakh Cyrillic with mixed content
print("Example 1: Kazakh mixed script and diacritics")
print("-" * 50)
kaz_text = "Мен Алматыда университетте оқимын."
print(f"Input:  {kaz_text}")
result = normalize_turkic_text(kaz_text, target_script=Script.LATIN, lang="kaz")
print(f"Output: {result}")
print()

# Example 2: Uzbek text
print("Example 2: Uzbek Cyrillic")
print("-" * 50)
uzb_text = "Ўзбекистонда қандай обўнашкан қўрылмалар бор?"
print(f"Input:  {uzb_text}")
result = normalize_turkic_text(uzb_text, target_script=Script.LATIN, lang="uzb")
print(f"Output: {result}")
print()

## 6. Cross-Script Text Processing

Demonstrate how to build a script-agnostic NLP pipeline that processes text in any script.

In [ ]:
from turkicnlp import Pipeline

def process_any_script(text: str, lang: str, processors: list):
    """
    Create a script-agnostic pipeline for Turkic text.
    Automatically detects script and processes accordingly.
    """
    # Detect original script
    original_script = detect_script(text)
    print(f"Detected script: {original_script.name}")
    
    # If Cyrillic, transliterate to Latin for processing
    if original_script == Script.CYRILLIC:
        t = Transliterator(lang, source=Script.CYRILLIC, target=Script.LATIN)
        text = t.transliterate(text)
        print(f"Transliterated to Latin for processing")
    
    # Create pipeline on Latin (most widely supported)
    nlp = Pipeline(lang, processors=processors, script="Latn")
    doc = nlp(text)
    return doc

# Example: Process Kazakh in both scripts with a Latin-trained model
print("Script-Agnostic Processing: Kazakh Tokenization")
print("=" * 50)

kaz_cyrl = "Мен барамын."
kaz_latn = "Men baramyn."

print(f"\nCyrillic input: {kaz_cyrl}")
doc_cyrl = process_any_script(kaz_cyrl, "kaz", ["tokenize"])
for tok in doc_cyrl.tokens:
    print(f"  Token: {tok.text}")

print(f"\nLatin input: {kaz_latn}")
doc_latn = process_any_script(kaz_latn, "kaz", ["tokenize"])
for tok in doc_latn.tokens:
    print(f"  Token: {tok.text}")
print()

## 7. Multi-Language Transliteration Comparison

Compare transliteration across different language-pair combinations.

In [ ]:
# Define test sentences in Cyrillic
test_sentences = {
    "kaz": "Алматы Қазақстанның ең үлкен қаласы болып табылады.",
    "uzb": "Ўзбекистон ўз ериман сўзи асосида озод давлатдир.",
    "tat": "Татарстан Россия Федерациясенең составында җәелгән бер республика.",
    "tuk": "Түркменистан борбор Азияны турганда барык түтүнчек берүүчі өлке.",
}

print("Multi-Language Transliteration Comparison")
print("=" * 60)
print(f"{'Language':<15} {'Cyrillic':<45} {'Latin'}")
print("-" * 60)

for lang, cyrl_text in test_sentences.items():
    try:
        t = Transliterator(lang, source=Script.CYRILLIC, target=Script.LATIN)
        latn_text = t.transliterate(cyrl_text)
        # Truncate for display
        cyrl_display = cyrl_text[:42] + "..." if len(cyrl_text) > 45 else cyrl_text
        latn_display = latn_text[:42] + "..." if len(latn_text) > 45 else latn_text
        print(f"{lang:<15} {cyrl_display:<45} {latn_display}")
    except Exception as e:
        print(f"{lang:<15} [Error: {str(e)[:35]}...]")
print()

## 8. Handling Special Cases

Demonstrate handling of edge cases and special characters.

In [ ]:
# Special case 1: Numbers and punctuation preservation
print("Special Case 1: Numbers and Punctuation Preservation")
print("=" * 50)

text_with_nums = "Алматы 2026 жылы тағдырының сынағынан өтеді!"
print(f"Input:  {text_with_nums}")

t = Transliterator("kaz", source=Script.CYRILLIC, target=Script.LATIN)
result = t.transliterate(text_with_nums)
print(f"Output: {result}")
print("✓ Numbers and punctuation preserved")
print()

# Special case 2: Proper noun capitalization
print("Special Case 2: Proper Noun Capitalization")
print("=" * 50)

text_proper = "Алмат Ершайле Қазақстанға барды."
print(f"Input:  {text_proper}")
result = t.transliterate(text_proper)
print(f"Output: {result}")
print("✓ Capitalization preserved")
print()

# Special case 3: Abbreviations and acronyms
print("Special Case 3: Abbreviations")
print("=" * 50)

text_abbr = "БҚҚ және ҚМ қызмет көрсетеді."
print(f"Input:  {text_abbr}")
result = t.transliterate(text_abbr)
print(f"Output: {result}")
print("✓ Acronyms handled")
print()

## 9. Practical Use Case: Building a Mixed-Script Corpus

Real-world example: processing a corpus that mixes Cyrillic and Latin Kazakh text.

In [ ]:
# Simulated mixed-script corpus (as would be found in web-crawled data)
mixed_corpus = [
    ("cyrl", "Алматы қаласы өте красивая."),
    ("latn", "Men Almaty shehrinde yashaymyn."),
    ("cyrl", "Қазақстан бүкіл дүниеге белгілі."),
    ("latn", "Qazaqstan butun duniyege belgili."),
    ("cyrl", "Университет 2020 жылы құрылды."),
]

print("Processing Mixed-Script Corpus")
print("=" * 60)

t = Transliterator("kaz", source=Script.CYRILLIC, target=Script.LATIN)

# Normalize all to Latin
normalized_corpus = []
for idx, (script_tag, text) in enumerate(mixed_corpus, 1):
    if script_tag == "cyrl":
        normalized = t.transliterate(text)
    else:
        normalized = text
    
    normalized_corpus.append(normalized)
    print(f"{idx}. [{script_tag.upper():4}] {text:<40} → {normalized}")

print(f"\n✓ All {len(normalized_corpus)} documents normalized to Latin script")
print()

## 10. Transliteration Coverage Summary

Supported language-script pairs in TurkicNLP.

In [ ]:
# Summary of supported transliteration pairs
coverage = {
    "Kazakh (kaz)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Official 2021 Kazakh Latin alphabet",
        "example": "Мен ↔ Men"
    },
    "Uzbek (uzb)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Official 1995 Uzbek Latin alphabet",
        "example": "Ўзбек ↔ Uzbek"
    },
    "Azerbaijani (aze)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Official 1991 Azerbaijani Latin",
        "example": "Азәрбајҹан ↔ Azərbaycən"
    },
    "Tatar (tat)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Zamanälif (new Tatar Latin)",
        "example": "Татар ↔ Tatar"
    },
    "Uyghur (uig)": {
        "direction": "Perso-Arabic ↔ Latin",
        "standard": "ULY (Uyghur Latin Yéziqi)",
        "example": "ئۇيغۇر ↔ Uyghur"
    },
    "Turkmen (tuk)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Official 1993 Turkmen Latin",
        "example": "Түркмен ↔ Türkmen"
    },
    "Karakalpak (kaa)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Official 2016 Karakalpak Latin",
        "example": "Қарақалпақ ↔ Qoraqalpoq"
    },
    "Crimean Tatar (crh)": {
        "direction": "Cyrillic ↔ Latin",
        "standard": "Crimean Tatar Latin standard",
        "example": "Крым ↔ Qırım"
    },
    "Ottoman Turkish (ota)": {
        "direction": "Perso-Arabic → Latin",
        "standard": "Academic transliteration convention",
        "example": "ترکی → Türkî"
    },
}

print("Transliteration Coverage in TurkicNLP")
print("=" * 80)
print(f"{'Language':<25} {'Direction':<25} {'Standard':<30}")
print("-" * 80)

for lang, info in coverage.items():
    print(f"{lang:<25} {info['direction']:<25} {info['standard']:<30}")

print(f"\n✓ Total coverage: {len(coverage)} language-script pairs")
print()

## Summary

You have learned how to:

1. **Detect scripts** automatically using TurkicNLP's script detection
2. **Transliterate bidirectionally** between Cyrillic and Latin (and Arabic and Latin)
3. **Handle Unicode edge cases** including the Turkish İ/I problem and Cyrillic/Latin homoglyphs
4. **Build complete preprocessing pipelines** combining multiple normalization steps
5. **Process mixed-script corpora** from web crawling or real-world text
6. **Create script-agnostic NLP pipelines** that work transparently across scripts

### Key Takeaways

- **Always normalize Unicode to NFC** before any other processing
- **Detect script early** in the pipeline to determine subsequent processing steps
- **Preserve transliteration directionality**: some pairs (Ottoman Turkish) are one-way only
- **Test transliteration on real data**: edge cases like diacritics, proper nouns, and abbreviations may not be perfect
- **For mixed-script corpora**, transliterate to a single target script before training models

### Further Reading

- Chapter 3 (Scripts, Encoding, and Orthographic Engineering) of the Turkic NLP textbook
- TurkicNLP documentation: https://turkic-nlp.github.io
- Unicode Standard, Section 14: Old Turkic (https://unicode.org)
- Apertium Turkic transducers: https://github.com/apertium/